## 1. Stack : 分散式平衡復原系統 - POP 操作

In [ ]:
# 114AB0041 林杰陞

def balanced_pop(stacks):
    # TODO 1: Empty 檢查
    if all(len(s) == 0 for s in stacks):
        print("Server Empty")
        return None, stacks

    # TODO 2: 寫一個迴圈找出長度「最長」的 stack 編號 (若同長度請取較小編號)
    max_len = 0
    target_idx = 0

    for i in range(len(stacks)):
        if len(stacks[i]) > max_len:
            max_len = len(stacks[i])
            target_idx = i

    # TODO 3: 將資料從找到的目標 Stack 中 Pop 出
    pop_data = stacks[target_idx].pop()

    return pop_data, stacks


# ----------------------------------------
# 測試區 : 測資 A
server_stacks = [['A', 'B'], ['C', 'D', 'E'], ['F']]
data, stacks = balanced_pop(server_stacks)
print("取出的資料:", data)
print("剩下的狀態:", stacks)

# ANS :
# Pop 出的資料: E
# 剩餘 Stack: [['A', 'B'], ['C', 'D'], ['F']]

## 2. Two-Stack : 圖書館自動化系統

In [ ]:
# 114AB0041 林杰陞

from collections import deque

def library_process(s1, s2, q, action, book=None):
    # TODO 1
    if action == "RETURN":
        s1.append(book)

    elif action == "SHELF":
        # TODO 2: 檢查是否 "No books to shelf"
        if len(s1) == 0 and len(s2) == 0:
            print("No books to shelf")
            return

        # TODO 3: 當整理車 (s2) 為「空」時，將還書箱 (s1) 的書全部搬移過去
        if len(s2) == 0:
            while len(s1) > 0:
                s2.append(s1.pop())

        # TODO 4: 從整理車 (s2) 拿出一本書，放到書架 (q) 上，並印出 "上架成功：{book_to_shelf}"
        book_to_shelf = s2.pop()
        q.append(book_to_shelf)
        print(f"上架成功：{book_to_shelf}")


# ----------------------------------------
# 測試區 : 測資 A
s1 = []     # 還書箱
s2 = []     # 整理車
q = deque() # 書架

library_process(s1, s2, q, "RETURN", "Python入門")
library_process(s1, s2, q, "RETURN", "資料結構")
library_process(s1, s2, q, "SHELF")
library_process(s1, s2, q, "SHELF")
print(f"測資 A 書架最終清單: {list(q)}")

# ANS :
# ['Python入門', '資料結構']

## 3. 演唱會多開雙層售票系統 (負載平衡 + Two-Stack Priority)

In [ ]:
# 114AB0041 林杰陞

def ticket_system(counters, action, user=None, q_idx=None):
    # 客人抵達
    if action == "ARRIVE":
        min_len = 999
        target_idx = -1

        # TODO 1: 尋找最少人的窗口
        for i in range(len(counters)):
            total = len(counters[i]["s1"]) + len(counters[i]["s2"])
            if total < min_len:
                min_len = total
                target_idx = i

        if not user.startswith("VIP"):
            # TODO 2: 一般客人排入找到的 target_idx 窗口
            counters[target_idx]["s1"].append(user)
            print(f"一般客 {user} 排入窗口 {target_idx}")

        elif user.startswith("VIP"):
            # TODO 3: VIP 客人排入 target_idx 窗口，且必須是「下一個」被服務的人
            counters[target_idx]["s2"].append(user)
            print(f"VIP {user} 霸道插隊進入窗口 {target_idx} 最前方！")

    # 窗口服務客人
    elif action == "SERVE":
        c = counters[q_idx]

        # TODO 4: 檢查該窗口是否全空 (Underflow 保護)
        if len(c["s1"]) == 0 and len(c["s2"]) == 0:
            print(f"窗口 {q_idx} 目前沒有客人！")
            return

        # TODO 5: Two-Stack 倒裝與出列邏輯 (從 target_idx 窗口服務客人)
        if len(c["s2"]) == 0:
            while len(c["s1"]) > 0:
                c["s2"].append(c["s1"].pop())

        served_user = c["s2"].pop()
        print(f"窗口 {q_idx} 服務完成：{served_user}")


# ----------------------------------------
# 測試區 : 測資 A
counters = [
    {"s1": [], "s2": []},
    {"s1": [], "s2": []},
    {"s1": [], "s2": []}
]

ticket_system(counters, "ARRIVE", "A")
ticket_system(counters, "ARRIVE", "B")
ticket_system(counters, "ARRIVE", "C")
ticket_system(counters, "ARRIVE", "D")
ticket_system(counters, "ARRIVE", "E")
ticket_system(counters, "ARRIVE", "F")
ticket_system(counters, "SERVE", q_idx=0)

# ANS :
# 一般客 A 排入窗口 0
# 一般客 B 排入窗口 1
# 一般客 C 排入窗口 2
# 一般客 D 排入窗口 0
# 一般客 E 排入窗口 1
# 一般客 F 排入窗口 2
# 窗口 0 服務完成：A

## 4. 雲霄飛車系統 (Queue + 動態耐心值)

In [ ]:
# 114AB0041 林杰陞

from collections import deque

def theme_park_system(q, action, name=None, patience=None, batch_size=None):
    if action == "JOIN":
        q.append((name, patience))

    elif action == "RIDE":
        if batch_size is None:
            batch_size = name
        boarded = []

        for _ in range(batch_size):
            if len(q) == 0:
                break

            rider_name, rider_patience = q.popleft()
            boarded.append(rider_name)

            # 用純 Queue 操作 (popleft/append) 更新耐心值，不使用迭代或索引
            remaining = len(q)
            status_parts = []
            for _ in range(remaining):
                pname, ppat = q.popleft()
                ppat -= 1
                if ppat > 0:
                    q.append((pname, ppat))
                    status_parts.append(f"{pname}(耐心剩{ppat})")

            if status_parts:
                print(f"{rider_name} 上車。{', '.join(status_parts)}")
            else:
                print(f"{rider_name} 上車。")

        print(f"==> 飛車發車！本次乘客: {boarded}")
        return boarded


# ----------------------------------------
# 測試區 : 測資 A
q = deque() # 純 Queue 宣告

theme_park_system(q, "JOIN", "Alice", 5)
theme_park_system(q, "JOIN", "Bob", 5)
theme_park_system(q, "JOIN", "Charlie", 5)
theme_park_system(q, "RIDE", 2)

# ANS :
# Alice 上車。Bob(耐心剩4), Charlie(耐心剩4)
# Bob 上車。Charlie(耐心剩3)
# 最終發車載了: ['Alice', 'Bob']

## 5. 思考題 : Debug Deque

**114AB0041 林杰陞**

### 答案：證明 deque([4, 2, 1, 5, 3]) 不可能由依序插入 1~5 產生

**核心觀察：deque 只有 `appendleft()`（插入左端）和 `append()`（插入右端）兩種操作。**

每次新元素只能放在「最左邊」或「最右邊」，這代表每次插入後，新元素一定會出現在 deque 的**頭部或尾部**。

我們按照 1 → 2 → 3 → 4 → 5 的順序逐步分析：

---

**第 1 步：插入 1**
- 只有一種可能：`deque([1])`

**第 2 步：插入 2**
- `appendleft(2)` → `deque([2, 1])`
- `append(2)` → `deque([1, 2])`

**第 3 步：插入 3**
- 3 一定在最左或最右，所以 3 一定出現在 deque 的**第一個位置**或**最後一個位置**。

**第 4 步：插入 4**
- 同理，4 一定在最左或最右。

**第 5 步：插入 5**
- 同理，5 一定在最左或最右。

---

**關鍵推論：最後兩個被插入的元素（4 和 5）一定分別佔據 deque 的頭部或尾部。**

因為第 5 步插入的 5 一定在最外層（最左或最右），第 4 步插入的 4 在插入時也在最外層，而 5 只能被放在 4 的外側或另一側。

所以最終 deque 的**第一個元素和最後一個元素，一定是 4 和 5 的某種組合**（即 {首, 尾} ⊆ {4, 5} 或其中一個是 4 或 5）。

更精確地說：
- 5 一定在位置 0 或位置 4（最左或最右）
- 觀察目標 `deque([4, 2, 1, 5, 3])`：5 在**位置 3**（index 3），既不是第一個也不是最後一個！

**這在物理上是不可能的。** 因為 5 是最後一個被插入的，它只能透過 `appendleft()` 或 `append()` 插入，所以它一定會在 deque 的最左端（index 0）或最右端（index 4）。但在 `deque([4, 2, 1, 5, 3])` 中，5 出現在 index 3，這代表 5 被插入後還有其他元素被放到它的右邊（3 在 5 的右邊），但 5 是最後插入的，之後不可能再有元素插入。

**結論：`deque([4, 2, 1, 5, 3])` 不可能是按照 1→2→3→4→5 的順序，透過 `appendleft()` 和 `append()` 操作產生的。小明說謊了。**

## 6. 思考題 : Deque 的效能災難

**114AB0041 林杰陞**

### 答案：為什麼把 list 換成 deque 後，反轉陣列的程式會卡到當機？

小明的反轉程式碼如下：

```python
def reverse_data(q):  # q 是一個長度為 N 的 deque
    left = 0
    right = len(q) - 1
    while left < right:
        q[left], q[right] = q[right], q[left]
        left += 1
        right -= 1
```

---

#### 根本原因：`deque` 的「隨機索引存取」是 O(n)，而 `list` 是 O(1)

**Python list（Dynamic Array）的運作原理：**

Python 的 `list` 底層是一個**連續記憶體的動態陣列（Dynamic Array）**。所有元素在記憶體中是連續排列的，因此透過索引存取 `list[i]` 時，只需要計算 `起始位址 + i × 元素大小`，就能直接跳到目標位置。這個操作的時間複雜度是 **O(1)**，無論陣列多大，存取速度都一樣快。

**Python deque（Doubly Linked List）的運作原理：**

Python 的 `collections.deque` 底層是一個**雙向鏈結串列（Doubly Linked List）**（嚴格來說是 block-based doubly linked list）。元素分散在記憶體各處，每個節點只知道「前一個」和「下一個」節點的位址。因此，當你用索引存取 `deque[i]` 時，deque **無法直接跳到第 i 個位置**，它必須從頭部或尾部開始，沿著鏈結一個節點一個節點地「走」過去，直到走到第 i 個位置。這個操作的時間複雜度是 **O(n)**。

---

#### 效能分析

在反轉演算法中，迴圈跑了 `N/2` 次，每次迴圈做了兩次索引存取（`q[left]` 和 `q[right]`），加上兩次索引賦值。

| 資料結構 | 每次索引存取 | 迴圈次數 | 總時間複雜度 |
|---------|-----------|---------|-----------|
| **list** | O(1) | N/2 | **O(n)** |
| **deque** | O(n) | N/2 | **O(n²)** |

當 N = 100,000 時：
- **list 版本**：大約 100,000 次操作 → 瞬間完成
- **deque 版本**：大約 100,000 × 100,000 = **100 億次操作** → 卡到當機

---

#### 結論

deque 的優勢在於**頭尾的新增/刪除是 O(1)**（`append`、`appendleft`、`pop`、`popleft`），但它的**隨機索引存取 `deque[i]` 是 O(n)**。小明無腦把所有 list 替換成 deque 是錯誤的做法，因為任何需要「用索引存取中間元素」的演算法（如這個反轉陣列），換成 deque 後效能都會從 O(n) 暴跌到 O(n²)。

**正確做法：** 根據使用場景選擇資料結構。需要頻繁在頭尾操作 → 用 `deque`；需要頻繁用索引隨機存取 → 用 `list`。

## 7. 特殊題 : 伺服器 bug 還原

In [ ]:
# 114AB0041 林杰陞

"""
答案 1：給 AI 的 Prompt

「請幫我寫一隻 Python 程式，功能如下：
1. 讀取一個名為 server_log.txt 的日誌檔案，每一行的格式為：
   [時間戳記] 操作 [參數]
   例如：[000001] PUSH 1045、[000003] POP、[000005] REVERSE_ALL
2. 使用 collections.deque 模擬一個 Queue 資料結構
3. 支援三種操作：
   - PUSH <number>：將數字加入 Queue 尾端
   - POP：從 Queue 前端移除一個元素
   - REVERSE_ALL：將整個 Queue 反轉
4. 執行完所有操作後，印出 Queue 中「最後 5 個」元素（即 Queue 尾端的 5 個訂單編號）」
"""

from collections import deque

def simulate_server_log(filename):
    q = deque()

    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split('] ')
            if len(parts) < 2:
                continue
            command_part = parts[1].strip()
            tokens = command_part.split()
            operation = tokens[0]

            if operation == "PUSH":
                value = int(tokens[1])
                q.append(value)
            elif operation == "POP":
                if len(q) > 0:
                    q.popleft()
            elif operation == "REVERSE_ALL":
                q.reverse()

    result = list(q)
    last_5 = result[-5:] if len(result) >= 5 else result
    print(f"目前隊伍中最後 5 個訂單編號: {last_5}")
    print(f"隊伍總長度: {len(q)}")
    return last_5

# 執行
answer2 = simulate_server_log("server_log.txt")

# 答案 2：最後 5 個訂單編號為 [600702, 600703, 600704, 600705, 600740]

## 8. 特殊題 : 用 AI 視覺化 Two-Stack 流程

In [ ]:
# 114AB0041 林杰陞

"""
答案：請開啟同目錄下的 two-stack-visualizer/index.html

功能說明：
- 分離式架構：HTML + CSS + JS 三個檔案
- 兩個視覺容器：Stack 1（進件區）與 Stack 2（出件區）
- PUSH 按鈕：產生帶有數字的彩色方塊，推入 Stack 1
- POP 按鈕：從 Stack 2 取出元素；若 Stack 2 為空，先動畫搬移 Stack 1 全部倒入 Stack 2，再取出
- 亮色/暗色主題切換（平滑過渡）
- 中文/英文語言切換（即時 i18n）
- 實現真正的 Two-Stack Queue 流程
- 包含動態動畫效果與操作日誌
- 鍵盤快捷鍵：P = PUSH, O = POP
"""

import webbrowser
import os

html_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "two-stack-visualizer", "index.html")
if os.path.exists(html_path):
    webbrowser.open("file://" + os.path.abspath(html_path))
    print("已在瀏覽器中開啟 Two-Stack 視覺化！")
else:
    print(f"請確認檔案存在於: {html_path}")